## Tarea Clasificación
### Materia: Laboratorio de Aprendizaje Estádistico 
### Profesor: Antonio Manguart Paez
### Alumna: Maria Fernanda Gómez Flores


Convertir el precio de las casas a clasificación

1 = casas con precio mayor a 250000  
0 = casas con precio menor o igual a 250000

Crear:
- KNN
- Regresión logística
- Polinomial
- Ridge
- LDA

Obtener el modelo con mayor AUC

Realizar calibración


## Preparación de datos


In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import linear_model


In [2]:
df = pd.read_csv('housing.csv')
df.head()


FileNotFoundError: [Errno 2] No such file or directory: 'housing.csv'

In [ ]:
df.isna().mean()


In [ ]:
df = df.fillna(df.mean(numeric_only=True))


In [ ]:
df.isna().mean()


## Target


In [ ]:
df['target'] = df['median_house_value'] > 250000


In [ ]:
df.target.mean()


El target es 1 cuando la casa vale mas de 250000 y 0 cuando vale 250000 o menos


In [ ]:
y = df['target']

X = df.copy()
del X['median_house_value']
del X['target']


ocean_proximity es categorica entonces la convierto a dummies


In [ ]:
categoricas = [
    'ocean_proximity'
]


In [ ]:
X = pd.get_dummies(X, columns=categoricas)


In [ ]:
X.head()


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0)


## Regresión Logística


In [ ]:
from sklearn.linear_model import LogisticRegression


In [ ]:
model_logistica = LogisticRegression(max_iter=2000)
model_logistica.fit(X_train,y_train)


In [ ]:
probabilidades_logistica = model_logistica.predict_proba(X_test)[:,1]
probabilidades_logistica


In [ ]:
from sklearn.metrics import roc_auc_score

auc_logistica = roc_auc_score(y_score=probabilidades_logistica, y_true=y_test)
auc_logistica


In [ ]:
from sklearn.metrics import roc_curve

fpr_logistica, tpr_logistica, umbrales = roc_curve(
    y_true=y_test,
    y_score=probabilidades_logistica
)


In [ ]:
plt.plot(fpr_logistica,tpr_logistica)
plt.plot([0, 1], [0, 1], color='red', linestyle='--', lw=2, label='Clasificador aleatorio')
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.grid()


El AUC de la regresión logística es aproximadamente 0.915 por lo que logra separar bastante bien las dos clases


## KNN


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


In [ ]:
model_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('model', KNeighborsClassifier(n_neighbors=5))
])

model_knn.fit(X_train,y_train)


In [ ]:
probabilidades_knn = model_knn.predict_proba(X_test)[:,1]
probabilidades_knn


In [ ]:
auc_knn = roc_auc_score(y_score=probabilidades_knn, y_true=y_test)
auc_knn


In [ ]:
fpr_knn, tpr_knn, umbrales = roc_curve(
    y_true=y_test,
    y_score=probabilidades_knn
)


In [ ]:
plt.plot(fpr_knn,tpr_knn)
plt.plot([0, 1], [0, 1], color='red', linestyle='--', lw=2, label='Clasificador aleatorio')
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.grid()


El KNN usa los vecinos mas cercanos para clasificar. Se escalaron las variables porque trabaja con distancias


## Polinomial


In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV


In [ ]:
pipeline = Pipeline([
    ('poly', PolynomialFeatures()),
    ('scaler', StandardScaler()),
    ('model', linear_model.LogisticRegression())
])


In [ ]:
params = {
    'poly__degree': [2]
}


In [ ]:
grid = GridSearchCV(
    estimator=pipeline,
    param_grid=params,
    scoring='roc_auc',
    cv=5
)


In [ ]:
grid.fit(X_train,y_train)


In [ ]:
grid.best_params_


In [ ]:
probabilidades_poly = grid.predict_proba(X_test)[:,1]
probabilidades_poly


In [ ]:
auc_poly = roc_auc_score(y_score=probabilidades_poly, y_true=y_test)
auc_poly


In [ ]:
fpr_poly, tpr_poly, umbrales = roc_curve(
    y_true=y_test,
    y_score=probabilidades_poly
)


In [ ]:
plt.plot(fpr_poly,tpr_poly)
plt.plot([0, 1], [0, 1], color='red', linestyle='--', lw=2, label='Clasificador aleatorio')
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.grid()


El modelo polinomial agrega relaciones de grado 2 entre las variables para ver si mejora la clasificación


## Ridge


In [ ]:
model_ridge = Pipeline([
    ('scaler', StandardScaler()),
    ('model', linear_model.LogisticRegression(penalty='l2'))
])

model_ridge.fit(X_train,y_train)


In [ ]:
probabilidades_ridge = model_ridge.predict_proba(X_test)[:,1]
probabilidades_ridge


In [ ]:
auc_ridge = roc_auc_score(y_score=probabilidades_ridge, y_true=y_test)
auc_ridge


In [ ]:
fpr_ridge, tpr_ridge, umbrales = roc_curve(
    y_true=y_test,
    y_score=probabilidades_ridge
)


In [ ]:
plt.plot(fpr_ridge,tpr_ridge)
plt.plot([0, 1], [0, 1], color='red', linestyle='--', lw=2, label='Clasificador aleatorio')
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.grid()


Ridge usa penalización L2 para controlar los coeficientes del modelo


## LDA


In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis


In [ ]:
model_lda = LinearDiscriminantAnalysis()
model_lda.fit(X_train,y_train)


In [ ]:
probabilidades_lda = model_lda.predict_proba(X_test)[:,1]
probabilidades_lda


In [ ]:
auc_lda = roc_auc_score(y_score=probabilidades_lda, y_true=y_test)
auc_lda


In [ ]:
fpr_lda, tpr_lda, umbrales = roc_curve(
    y_true=y_test,
    y_score=probabilidades_lda
)


In [ ]:
plt.plot(fpr_lda,tpr_lda)
plt.plot([0, 1], [0, 1], color='red', linestyle='--', lw=2, label='Clasificador aleatorio')
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.grid()


LDA busca una combinación de las variables que permita separar las dos clases


## Comparación de modelos


In [ ]:
resultados = pd.DataFrame({
    'Modelo': ['Regresión Logística','KNN','Polinomial','Ridge','LDA'],
    'AUC': [auc_logistica,auc_knn,auc_poly,auc_ridge,auc_lda]
})

resultados = resultados.sort_values(by='AUC',ascending=False)
resultados


In [ ]:
plt.figure(figsize=(10,6))

plt.plot(fpr_logistica,tpr_logistica,label="Regresión Logística")
plt.plot(fpr_knn,tpr_knn,label="KNN")
plt.plot(fpr_poly,tpr_poly,label="Polinomial")
plt.plot(fpr_ridge,tpr_ridge,label="Ridge")
plt.plot(fpr_lda,tpr_lda,label="LDA")

plt.plot([0, 1], [0, 1], color='red', linestyle='--', lw=2, label='Clasificador aleatorio')
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.legend()
plt.grid()


El mejor modelo fue el polinomial con un AUC de aproximadamente 0.9374 por lo que fue el que mejor logro separar las casas que cuestan mas de 250000 de las que cuestan 250000 o menos


## Calibración


In [ ]:
df_calibracion = pd.DataFrame({
    'y': y_test,
    'probabilidad': probabilidades_poly
})


In [ ]:
df_calibracion['probabilidad_bin'] = pd.qcut(
    df_calibracion['probabilidad'],
    q=10,
    labels=False
) + 1


In [ ]:
calibracion = df_calibracion.groupby('probabilidad_bin').mean()
calibracion


In [ ]:
plt.plot(calibracion.probabilidad, calibracion.y, marker="o")
plt.plot([0,1],[0,1],"--")
plt.xlabel("Probabilidad predicha")
plt.ylabel("Real")
plt.grid()


La calibración compara la probabilidad que predice el modelo contra lo que realmente paso

Si los puntos estan cerca de la diagonal significa que las probabilidades del modelo estan bien calibradas


## Conclusión


Se compararon los 5 modelos usando AUC porque permite medir que tan bien separan las dos clases

El polinomial fue el que obtuvo el AUC mas alto con aproximadamente 0.9374 por lo que fue el mejor modelo de los que se probaron

Despues se reviso su calibración comparando las probabilidades predichas contra la proporción real de casas mayores a 250000
